# EA3 — Comparación climática: Medellín y Ciudad de México (2024)

**Asignatura:** Análisis de datos  
**Estudiante:** Nicolás Restrepo  
**Modalidad:** Individual  
**Periodo de estudio:** 1 de enero a 31 de diciembre de 2024 (366 días, año bisiesto).

## Pregunta guía
¿Qué similitudes y diferencias se observan en las variables climáticas de Medellín (Colombia) y Ciudad de México (México) durante el mismo periodo?

## A. Planteamiento e hipótesis previa
Se espera que Medellín muestre una temperatura máxima diaria relativamente estable durante el año y que Ciudad de México tenga un contraste mensual más marcado. Ambas ciudades tendrán días de precipitación, pero posiblemente con diferentes acumulados y estacionalidad. Estas **son predicciones, no resultados**.

**Fuente para ambas tablas:** [Open-Meteo, Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api), reanálisis histórico **ERA5**, variable diaria local. La API permite especificar fechas, zonas horarias, coordenadas y el mismo modelo para ambas ciudades. Las direcciones **exactas** de extracción se generan e imprimen a continuación.

| Tabla | Coordenadas aproximadas | Zona horaria | Frecuencia | Rango | Variables y unidades |
|---|---|---|---|---|---|
| Medellín, Colombia | 6.2442, -75.5812 | America/Bogota | Diaria | 2024-01-01 a 2024-12-31 | temperatura máxima y mínima del aire a 2 m (°C), precipitación diaria (mm) |
| Ciudad de México, México | 19.4326, -99.1332 | America/Mexico_City | Diaria | 2024-01-01 a 2024-12-31 | temperatura máxima y mínima del aire a 2 m (°C), precipitación diaria (mm) |

**Comparabilidad:** idéntico calendario y frecuencia, modelo ERA5 y unidades. No es una comparación de mediciones de estaciones: la fuente entrega datos **modelados de reanálisis**; son estimaciones para cada coordenada y no representan toda el área metropolitana. Las fechas corresponden al día civil local de cada ciudad. Se usará el mismo umbral de **28 °C** para ambas ciudades. No se imputarán los faltantes ni se transformarán en días secos. Se conservarán los 366 días en los CSV y se usarán solo fechas con ambas variables clave completas en ambos lugares.

In [ ]:
from pathlib import Path
import json
import urllib.parse
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

BASE = Path.cwd()
if BASE.name != 'EA3_comparacion_climatica' and (BASE / 'EA3_comparacion_climatica').is_dir():
    BASE = BASE / 'EA3_comparacion_climatica'
DATOS = BASE / 'datos'
DATOS.mkdir(parents=True, exist_ok=True)
(BASE / 'graficas').mkdir(parents=True, exist_ok=True)
INICIO, FIN = '2024-01-01', '2024-12-31'
VARIABLES = ['temperature_2m_max', 'temperature_2m_min', 'precipitation_sum']
CIUDADES = {
    'Medellín': {'latitude': 6.2442, 'longitude': -75.5812, 'timezone': 'America/Bogota', 'archivo': 'medellin.csv'},
    'Ciudad de México': {'latitude': 19.4326, 'longitude': -99.1332, 'timezone': 'America/Mexico_City', 'archivo': 'ciudad_mexico.csv'}
}
print('Carpeta de trabajo:', BASE.resolve())

## B. Carga, calidad y cobertura de las dos tablas

Si los archivos CSV no están disponibles en `datos/`, la primera ejecución los **descarga automáticamente** de Open-Meteo. Se guardan de manera permanente en la carpeta de la actividad para que las siguientes ejecuciones sean reproducibles y no dependan de la conexión. Se comprueba que la API informe las unidades correctas. Los faltantes permanecen como `NaN`: un valor `0 mm` solo equivale a un día sin precipitación cuando procede de la fuente.

Se inspeccionan columnas, filas iniciales, dimensiones, fechas, duplicados y datos faltantes. Una comparación parcial de periodos se considera inválida y detiene la ejecución.

In [ ]:
def url_fuente(info):
    parametros = {
        'latitude': info['latitude'], 'longitude': info['longitude'],
        'start_date': INICIO, 'end_date': FIN,
        'daily': ','.join(VARIABLES), 'timezone': info['timezone'],
        'models': 'era5', 'temperature_unit': 'celsius', 'precipitation_unit': 'mm'
    }
    return 'https://archive-api.open-meteo.com/v1/archive?' + urllib.parse.urlencode(parametros)

def obtener_tabla(ciudad, info):
    ruta = DATOS / info['archivo']
    url = url_fuente(info)
    print(f'{ciudad} — Fuente: {url}')
    if not ruta.exists():
        try:
            with urllib.request.urlopen(url, timeout=90) as respuesta:
                contenido = json.load(respuesta)
        except Exception as exc:
            raise RuntimeError(f'No se pudo descargar {ciudad}. Conéctate a internet y ejecuta de nuevo.') from exc
        if 'daily' not in contenido:
            raise ValueError(f'La API no entregó datos diarios de {ciudad}: {contenido}')
        unidades = contenido.get('daily_units', {})
        assert unidades.get('temperature_2m_max') == '°C', unidades
        assert unidades.get('temperature_2m_min') == '°C', unidades
        assert unidades.get('precipitation_sum') == 'mm', unidades
        tabla = pd.DataFrame(contenido['daily'])
        tabla = tabla.rename(columns={'time': 'fecha'})
        tabla.to_csv(ruta, index=False)
        print('Descargado:', ruta)
    tabla = pd.read_csv(ruta)
    tabla['fecha'] = pd.to_datetime(tabla['fecha'], errors='coerce')
    for columna in VARIABLES:
        tabla[columna] = pd.to_numeric(tabla[columna], errors='coerce')
    return tabla

tablas = {nombre: obtener_tabla(nombre, info) for nombre, info in CIUDADES.items()}
for nombre, tabla in tablas.items():
    print('\n', nombre, '— dimensiones:', tabla.shape)
    print('Columnas:', tabla.columns.tolist())
    display(tabla.head(4))
    print('Fechas:', tabla.fecha.min(), 'a', tabla.fecha.max())
    print('Faltantes por columna:\n', tabla.isna().sum().to_string())
    print('Fechas duplicadas:', tabla.fecha.duplicated().sum())

In [ ]:
fechas_esperadas = pd.date_range(INICIO, FIN, freq='D')
for ciudad, tabla in tablas.items():
    assert not tabla['fecha'].isna().any(), f'Fecha inválida: {ciudad}'
    assert not tabla['fecha'].duplicated().any(), f'Fechas duplicadas: {ciudad}'
    assert tabla['fecha'].is_monotonic_increasing, f'Fechas desordenadas: {ciudad}'
    assert tabla['fecha'].tolist() == fechas_esperadas.tolist(), f'Cobertura diaria incompleta: {ciudad}'
    assert not (tabla['precipitation_sum'].dropna() < 0).any(), f'Precipitación negativa: {ciudad}'
print('Verificación aprobada: ambas tablas contienen exactamente las mismas 366 fechas diarias.')

# Unión por fecha y comparación de pares completos sin sustituir faltantes por cero.
m = tablas['Medellín'].set_index('fecha')[VARIABLES].add_suffix('_med')
c = tablas['Ciudad de México'].set_index('fecha')[VARIABLES].add_suffix('_mex')
pares = m.join(c, how='inner')
columnas_clave = ['temperature_2m_max_med','precipitation_sum_med',
                  'temperature_2m_max_mex','precipitation_sum_mex']
validos = pares.dropna(subset=columnas_clave).copy()
print('Fechas comunes:', len(pares))
print('Pares completos para temperatura máxima y precipitación:', len(validos))
print('Pares excluidos por faltantes:', len(pares)-len(validos))
print('Faltantes por ciudad y variable:\n', pares.isna().sum().to_string())
assert len(validos) > 0, 'No hay fechas válidas para comparar'
display(validos.head())

## C. NumPy: selección, máscaras, broadcasting y estadísticos

Los arrays se construyen a partir de **pares diarios completos**, alineados por fecha. Se usa un corte `[:7]` para inspeccionar la primera semana válida, máscaras `precipitación > 0 mm` y `temperatura máxima > 28 °C`, y un **broadcasting** de una matriz de dos columnas contra el escalar `28`. No se convierte la precipitación faltante a cero. Los máximos y mínimos de temperatura se refieren a **temperaturas máximas diarias**, no a temperaturas mínimas nocturnas.

In [ ]:
fecha = validos.index.to_numpy()
t_med = validos['temperature_2m_max_med'].to_numpy(dtype=float)
t_mex = validos['temperature_2m_max_mex'].to_numpy(dtype=float)
p_med = validos['precipitation_sum_med'].to_numpy(dtype=float)
p_mex = validos['precipitation_sum_mex'].to_numpy(dtype=float)

# Indexación y cortes NumPy
print('Siete primeras fechas válidas (corte de arrays):')
for f, tm, tx, pm, px in zip(fecha[:7], t_med[:7], t_mex[:7], p_med[:7], p_mex[:7]):
    print(str(f)[:10], f'Medellín: {tm:.1f} °C / {pm:.1f} mm',
          f'Ciudad de México: {tx:.1f} °C / {px:.1f} mm')

UMBRAL = 28.0
lluvia_med, lluvia_mex = p_med > 0, p_mex > 0
# Broadcasting: matriz (N, 2) versus un único umbral escalar
matriz_temperaturas = np.column_stack([t_med, t_mex])
mascara_calor = matriz_temperaturas > UMBRAL
calor_med, calor_mex = mascara_calor[:, 0], mascara_calor[:, 1]
# Operaciones vectorizadas y máscaras combinadas
diferencia_diaria = matriz_temperaturas[:, 0] - matriz_temperaturas[:, 1]
ambas_lluvia = lluvia_med & lluvia_mex
solo_una_lluvia = lluvia_med ^ lluvia_mex

def estadisticas(temperatura, lluvia, mascara_lluvia, mascara_calor):
    return {
        'T máxima media (°C)': np.mean(temperatura),
        'Menor T máxima diaria (°C)': np.min(temperatura),
        'Mayor T máxima diaria (°C)': np.max(temperatura),
        'Precipitación acumulada (mm)': np.sum(lluvia),
        'Días lluviosos (> 0 mm)': np.count_nonzero(mascara_lluvia),
        'Días con T máxima > 28 °C': np.count_nonzero(mascara_calor),
    }
resumen = pd.DataFrame({
    'Medellín': estadisticas(t_med, p_med, lluvia_med, calor_med),
    'Ciudad de México': estadisticas(t_mex, p_mex, lluvia_mex, calor_mex)
}).T
print('Días lluviosos simultáneos:', np.count_nonzero(ambas_lluvia))
print('Días con lluvia en una sola ciudad:', np.count_nonzero(solo_una_lluvia))
print(f'Diferencia media en T máxima diaria Medellín - CDMX: {np.mean(diferencia_diaria):+.2f} °C')
display(resumen.round(2))

## D. Visualización comparativa
Las agregaciones mensuales se calculan **únicamente sobre pares diarios completos**, para mantener la misma base de fechas en ambas ciudades. Si faltan días, las sumas mensuales son acumulados de días disponibles, **no necesariamente totales mensuales completos**. No se imputan valores.

In [ ]:
mensual = validos.groupby(validos.index.month).agg({
    'temperature_2m_max_med':'mean','temperature_2m_max_mex':'mean',
    'precipitation_sum_med':'sum','precipitation_sum_mex':'sum'
}).reindex(range(1,13))
conteo_mes = validos.groupby(validos.index.month).size().reindex(range(1,13),fill_value=0)
meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
print('Días válidos por mes:')
display(pd.DataFrame({'Mes':meses,'Días comparables':conteo_mes.values}))
display(mensual.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(11,5))
ax.plot(meses, mensual['temperature_2m_max_med'], marker='o', linewidth=2, label='Medellín')
ax.plot(meses, mensual['temperature_2m_max_mex'], marker='s', linewidth=2, label='Ciudad de México')
ax.set(title='Temperatura máxima diaria promedio por mes — 2024', xlabel='Mes', ylabel='Temperatura (°C)')
ax.legend(); ax.grid(alpha=.25); plt.tight_layout(); fig.savefig(BASE / 'graficas' / '01_temperatura_mensual.png', dpi=160, bbox_inches='tight'); plt.show()
mes_m = int(mensual['temperature_2m_max_med'].idxmax())
mes_c = int(mensual['temperature_2m_max_mex'].idxmax())
print(f'Observación: el mayor promedio mensual de T máxima en Medellín fue {mensual.loc[mes_m,"temperature_2m_max_med"]:.1f} °C ({meses[mes_m-1]}); en CDMX fue {mensual.loc[mes_c,"temperature_2m_max_mex"]:.1f} °C ({meses[mes_c-1]}).')
print('Límite: promedios mensuales no muestran episodios extremos dentro de cada mes ni prueban sus causas.')

In [ ]:
x = np.arange(12); ancho = .38
fig, ax = plt.subplots(figsize=(11,5))
ax.bar(x-ancho/2, mensual['precipitation_sum_med'], ancho, label='Medellín')
ax.bar(x+ancho/2, mensual['precipitation_sum_mex'], ancho, label='Ciudad de México')
ax.set_xticks(x,meses); ax.set(title='Precipitación acumulada mensual — 2024',xlabel='Mes',ylabel='Precipitación (mm)')
ax.legend(); ax.grid(axis='y',alpha=.2);plt.tight_layout(); fig.savefig(BASE / 'graficas' / '02_precipitacion_mensual.png', dpi=160, bbox_inches='tight'); plt.show()
mes_pm=int(mensual['precipitation_sum_med'].idxmax())
mes_pc=int(mensual['precipitation_sum_mex'].idxmax())
print(f'Observación: el mes con mayor precipitación registrada fue {meses[mes_pm-1]} en Medellín ({mensual.loc[mes_pm,"precipitation_sum_med"]:.1f} mm) y {meses[mes_pc-1]} en CDMX ({mensual.loc[mes_pc,"precipitation_sum_mex"]:.1f} mm).')
print('Límite: los acumulados no indican la intensidad horaria, ni permiten inferir por sí solos inundaciones o sus causas.')

In [ ]:
fig, ax=plt.subplots(figsize=(11,5))
bordes=np.arange(np.floor(min(t_med.min(),t_mex.min()))-1, np.ceil(max(t_med.max(),t_mex.max()))+2, 1)
ax.hist(t_med,bins=bordes,alpha=.6,label='Medellín')
ax.hist(t_mex,bins=bordes,alpha=.6,label='Ciudad de México')
ax.axvline(UMBRAL,linestyle='--',color='black',label='Umbral 28 °C')
ax.set(title='Distribución de temperaturas máximas diarias — 2024',xlabel='Temperatura máxima (°C)',ylabel='Número de días comparables')
ax.legend();plt.tight_layout(); fig.savefig(BASE / 'graficas' / '03_histograma_temperatura.png', dpi=160, bbox_inches='tight'); plt.show()
print(f'Observación: Medellín supera 28 °C en {calor_med.sum()} días y CDMX en {calor_mex.sum()} días, dentro de {len(validos)} fechas comparables.')
print('Límite: un histograma no conserva el orden temporal ni demuestra causalidad; solo describe frecuencias.')

## E. Interpretación y respuesta a la pregunta guía
Las siguientes celdas redactan hallazgos con **valores reales calculados**. Se incluye una similitud cuantificada y una diferencia, además de una advertencia sobre el alcance de los datos y sus faltantes.

In [ ]:
simultaneos = int(np.count_nonzero(ambas_lluvia))
media_med = float(np.mean(t_med)); media_mex = float(np.mean(t_mex))
prec_med = float(np.sum(p_med)); prec_mex = float(np.sum(p_mex))
print('HALLAZGO 1 — SIMILITUD:')
print(f'Ambas ciudades registraron precipitación durante 2024: Medellín tuvo {lluvia_med.sum()} días lluviosos y Ciudad de México {lluvia_mex.sum()}; en {simultaneos} fechas hubo lluvia en ambas.')
print('HALLAZGO 2 — DIFERENCIA:')
print(f'La temperatura máxima diaria media fue {media_med:.2f} °C en Medellín y {media_mex:.2f} °C en Ciudad de México; la diferencia Medellín − CDMX fue {media_med-media_mex:+.2f} °C.')
print(f'La precipitación acumulada en días comparables fue {prec_med:.1f} mm y {prec_mex:.1f} mm, respectivamente (diferencia {prec_med-prec_mex:+.1f} mm).')
print(f'Las fechas inicial y final son {validos.index.min().date()} y {validos.index.max().date()}; se compararon {len(validos)} de {len(pares)} días del año.')
print(f'Se excluyeron {len(pares)-len(validos)} fechas con al menos un dato clave faltante. Esto puede sesgar medias y subestimar acumulados si las ausencias no son aleatorias.')
print('INTERPRETACIÓN: los resultados describen asociaciones y patrones del año 2024 en datos de reanálisis ERA5; no demuestran causas ni representan necesariamente el clima de largo plazo.')
print('RESPUESTA: las dos ciudades comparten ocurrencia de lluvia, pero difieren en sus niveles medios de temperatura máxima y acumulados de precipitación, según los valores y meses mostrados.')

# Exportar un resumen verificable y cifras calculadas, sin inventar valores
informe = [
    'EA3: Medellín vs Ciudad de México — datos de reanálisis ERA5, 2024',
    f'Periodo fuente: {INICIO} a {FIN}; fechas originales: {len(pares)}',
    f'Días completos comparables: {len(validos)}; días excluidos: {len(pares)-len(validos)}',
    f'Medellín: T máxima diaria promedio {media_med:.2f} °C; lluvia {prec_med:.1f} mm; días lluviosos {lluvia_med.sum()}',
    f'Ciudad de México: T máxima diaria promedio {media_mex:.2f} °C; lluvia {prec_mex:.1f} mm; días lluviosos {lluvia_mex.sum()}',
    f'Días con lluvia en ambas ciudades: {simultaneos}',
    f'Diferencia media de T máxima diaria: {media_med-media_mex:+.2f} °C',
    'Advertencia: totales de precipitación se refieren solo a los días comparables.',
    'Los datos no prueban relaciones causales ni representan estaciones meteorológicas puntuales.'
]
(BASE / 'resumen_resultados.txt').write_text('\n'.join(informe), encoding='utf-8')
mensual.round(3).to_csv(BASE / 'resumen_mensual.csv', index_label='mes')
resumen.round(3).to_csv(BASE / 'resumen_estadistico.csv', index_label='ciudad')
print('Exportados: resumen_resultados.txt, resumen_mensual.csv, resumen_estadistico.csv y 3 PNG.')

## Conclusiones y reflexión metodológica

1. Comparar correctamente requiere la misma ventana temporal completa, la misma frecuencia, las mismas definiciones de variables y las mismas unidades. Los dos CSV deben contener **366 fechas diarias del año 2024**. El análisis solo utiliza fechas para las que ambas ciudades aportan temperatura máxima y precipitación válida; se informa el conteo.
2. Las máscaras, cortes y el broadcasting de NumPy permiten calcular estadísticos de forma vectorizada y transparente. Las tres visualizaciones describen tendencia mensual, acumulados y distribución de temperaturas.
3. **Similitud cuantitativa:** ambas ciudades presentan días con precipitación; la cantidad exacta de días simultáneos y días lluviosos de cada una aparece en el apartado E. **Diferencia cuantitativa:** los promedios de temperatura máxima, acumulados de lluvia y sus diferencias se muestran en el mismo apartado, respaldados por el resumen tabular y las figuras.
4. Los faltantes pueden reducir los días válidos y subestimar acumulados mensuales y anuales. Además, los datos de reanálisis tienen incertidumbre espacial y meteorológica. Un año no es suficiente para establecer conclusiones sobre el clima de largo plazo o sobre causalidad.

### Archivos producidos
La primera ejecución conectada descarga `datos/medellin.csv` y `datos/ciudad_mexico.csv`. También guarda tres figuras en `graficas/`, `resumen_estadistico.csv`, `resumen_mensual.csv` y `resumen_resultados.txt`. El notebook conserva tablas, resultados y gráficos en su salida al guardarlo tras la ejecución.

### Referencia y atribución
Open-Meteo. (s. f.). *Historical Weather API* [API de reanálisis ERA5]. https://open-meteo.com/en/docs/historical-weather-api  
Open-Meteo. (s. f.). *Historical Weather API specification*. https://github.com/open-meteo/open-meteo/blob/main/openapi/historical-weather.yml